# Irish Housing Density: Clustering Census 2022 Small Areas

## Setup

In [7]:
# Libraries
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

# Fixed random seed so results are the same every run
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Relative paths (notebook is in notebooks/, data is in data/)
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
FIG_DIR = Path("../figures")

SAPS_FILE = RAW_DIR / "SAPS_2022_Small_Area_UR_171024.csv"
BOUNDARY_FILE = RAW_DIR / "Small_Area_National_Statistical_Boundaries_2022_Ungeneralised_view_-4936014253783534521.csv"
GLOSSARY_FILE = RAW_DIR / "Glossary_Saps_2022_REVISED_21102024.xlsx"

# Check the raw files are where we expect
for f in [SAPS_FILE, BOUNDARY_FILE, GLOSSARY_FILE]:
    print(("OK      " if f.exists() else "MISSING ") + f.name)

# Print versions so the environment can be reproduced
print()
print("Python      :", sys.version.split()[0])
print("pandas      :", pd.__version__)
print("numpy       :", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("seaborn     :", sns.__version__)

OK      SAPS_2022_Small_Area_UR_171024.csv
OK      Small_Area_National_Statistical_Boundaries_2022_Ungeneralised_view_-4936014253783534521.csv
OK      Glossary_Saps_2022_REVISED_21102024.xlsx

Python      : 3.15.0b4
pandas      : 3.1.1
numpy       : 2.5.4
scikit-learn: 1.9.1
seaborn     : 0.13.2


# Phase 1: Business Understanding

## 1.1 Context and Problem
In recent years, Ireland has been having a massive population boom. Between 2016 and 2022, Ireland's population grew by 7.6% (361,671 people) [4]. About 171,000 of this increase was natural increase (births minus deaths) and about 190,000 was net migration, with people moving to Ireland from Europe and other continents for various reasons [4]. Over the same period the housing stock only grew by about 6% [4]. Put simply, Ireland wasn't able to keep up with the demand and there is currently a housing shortage.

As a solution, the government has a 'National Planning Framework' (NPF), which plans for around 50,000 additional households per year up to the year 2040 [3]. The NPF wants a share of new homes to be built inside the existing built-up footprint (the area already built on) of towns and cities [3, NPO 7–9, p. 22]:

- at least 40% of all new homes nationally
- at least 50% of the homes planned for the five cities
- at least 30% of the homes planned for other towns and villages

Building inside already built-up areas uses less land and makes services like buses, water and roads cheaper to provide. Outside of towns I have noticed a lot of spread-out one-off houses with big gaps between them, and the CSO data backs this up [6].

In this project, a small area is a CSO statistical unit: a block of roughly 80–120 homes. There are 18,919 small areas, and together they cover the whole country. The challenge for planners is that there is currently no easy way to compare all of these small areas and tell which types of area are low density but well placed for more homes.

## 1.2 Stakeholders

### Who supplies the data?

Two publicly available datasets are used:

- CSO (census data): small area population statistics from Census 2022, published by the CSO
- Tailte Éireann (boundaries and land area): small area boundaries made with the CSO and published in 2023 on Tailte Éireann's open data portal

The file details are in 2.1 Initial Data Collection Report.

### Who uses the results?

Government bodies such as county council (local authority) planners, the Department of Housing, Local Government and Heritage and the Land Development Agency would be able to use my data analysis for strategic planning of new housing developments.

### Who is affected?

People who need homes, such as renters, first-time buyers and young people living with their parents, are the main reason this project matters. They would benefit most from more homes being built in suitable areas.

Residents of areas which would be suitable for additional housing development would also be affected. The residents of these areas would see increased home building, and upon completion of these builds they could see an increase in population around their particular small area. These residents could benefit from more homes nearby and better local services or public transport as the area grows, but they could also risk crowding, extra pressure on schools and GPs, and change happening without local consultation.

## 1.3 Business (Practical) Objective

The goal of this project is to help planners compare all 18,919 small areas in Ireland and find the types of area that are low density but well placed for more homes, such as areas inside or close to existing towns and cities.

This would help planners in the following ways:

- County council (local authority) planners could narrow down which areas to look at first when planning new housing.
- The Department of Housing could compare areas across the whole country in the same way.
- The Land Development Agency could use the results when looking for land for new homes.

The results would also support the NPF target of building more homes inside existing built-up areas [3]. Areas with a lot of vacant homes could also be flagged, as reusing empty homes may be quicker than building new ones.

This project will not pick exact building sites. It is a starting point to help planners decide where to look.

## 1.4 Analytics Objective

### Task Type
I will be using clustering, which groups similar small areas together. There is no column in the data that says which areas are suitable for more housing, so clustering is used to find patterns without one.

### Unit of analysis
The unit of analysis is one small area. The census file [1] and the boundary file [2] are joined on the small area ID (GUID = SA_GUID_2022), giving 18,919 rows. The census file has one extra row, which is a total for all of Ireland and will be removed.

Small areas have a similar number of homes (usually 80–120), but their size varies a lot. A city small area can be smaller than 0.1 km², while a rural one is usually around 8 km². This is why homes per km² is used to measure density.

### Inputs
The following features will be included. All are percentages or rates so that small areas of different sizes can be compared:

- Homes per km² (housing density)
- % Flats/Apartments
- % built since 2011
- Vacancy rate
- Holiday home rate
- People per household
- % renting
- % public transport to work

The CSO urban/rural category is already in the census file [1] but will not be used as an input. It will be used after clustering to compare against my groups (5.3).

Before modelling, the features will be scaled so that homes per km² doesn't outweigh the others.

### Method
**Algorithm: K-means clustering**
K-means is an unsupervised algorithm. We pick the number of groups (k), and it puts small areas that look alike into the same group. I chose it because it is fast on a large dataset and each group is easy to describe using its average values. The number of groups will be chosen using the elbow method and silhouette score (4.1).

### Output
Each small area will get a group label. Expected groups could include:
- Dense city apartments with mostly renters
- Suburban housing estates
- Newer estates with many homes built since 2011
- Rural one-off houses with almost no public transport use
- Coastal or remote areas with many holiday homes

The most useful group for planners would be areas that are low density but close to existing towns, as these could be suitable for more homes. None of the inputs measures closeness to a town directly, so this will be found after clustering using the CSO urban/rural category. If most of a group's areas are "Rural areas with high urban influence" or "Satellite urban towns", that group is low density but near a town. Public transport use will also be checked as a second clue.

### Analytical Question
Can K-means clustering of Census 2022 small areas, using housing and household features, find groups of areas that are low density but close to existing towns and could be suitable for more homes?

## 1.5 Success Criteria

### Technical
The success criteria for the technical output are the following:
- **Silhouette score ≥ 0.25**: The silhouette score is calculated after clustering. It measures how close each area is to its own group compared with the next nearest group, on a scale from −1 (bad) to 1 (perfect).
- **Stability (Adjusted Rand Index ≥ 0.8)**: When K-means is re-run with different random starts, the groups match at least 80% between runs. This shows the groups are real and not down to luck.
- **Reproducibility**: Re-running the notebook with the same random seed gives identical results every time. This lets someone else check my work.
- **Group size ≥ 1% of areas**: No group is too small to be useful. Every group contains at least 1% of small areas (about 190).
- **Better than the baseline**: My groups add something beyond the CSO's 6 urban/rural categories, for example by splitting one category into areas that are clearly different (such as rural areas with high vacancy vs rural areas without). If the groups only repeated the CSO categories, the clustering would not have found anything new.

### Practical
The success criteria for the non-technical output are the following:
- **Every group can be described in one plain sentence**: Each group can be explained quickly using simple, non-technical language.
- **Results can be grouped by county**: Local authorities (councils) would use the results for local planning, so the results can be broken down by county.
- **At least one group identifies low-density areas near towns**: This is the main goal of the analysis. At least one group contains low-density areas near towns that could suit future housing. The other groups will be different types of area, such as city apartments or holiday-home areas.
- **Results are a shortlist for human review**: The results are presented as a shortlist for planners to review, not as automatic decisions. This matches 1.3.

## 1.6 Constraints, Assumptions and Risks

### Constraints

- **Open data only**: I only have access to openly available data from the CSO and Tailte Éireann. I cannot access local council data.
- **Missing information**: The data says nothing about zoning, land ownership, flood risk or water/wastewater capacity. These decide whether land can actually be built on. This is why the project can only shortlist area types, not pick exact sites (see 1.3).
- **Unable to analyse trends**: The datasets are a single snapshot from Census night (3 April 2022), so I cannot analyse how density has changed over time.
- **Time constraint**: The project is due on 4 December, so only one main method (K-means) and one alternative can be tested.

### Assumptions
- **Land area gives a fair comparison of density**: Homes per km² is gross density. The land area includes fields, rivers and mountains, not just land with homes on it, which makes rural areas look emptier than the land they actually build on. I assume it is still a fair way to compare areas, as long as this limitation is kept in mind.
- **2022 data is the best available**: The next census is in 2027, so 2022 is the most recent detailed data available for small areas. However, homes built since 2022 are not included.
- **The CSO urban/rural category is a fair measure of closeness to towns**: The category tells me how close an area is to a town, which is how I use it in 1.4. Density comes from my homes per km² feature, not this category. It may not be perfect, as it is the CSO's own classification and the boundaries between categories are a judgement call.

### Risks

| Risk | Why it matters | What I'll do (and when) |
|---|---|---|
| Choosing k | There is no single correct number of groups | Use the elbow method and silhouette score to pick a sensible k, and check that the groups make sense |
| Variation in homes per km² | It ranges from 0 to over 57,000. K-means uses distances, so huge values would dominate the groups | Log-transform homes per km² |
| % car and % public transport strongly overlap | They are strongly linked, so including both would count the same pattern twice | % car will be dropped in the data processing stage |
| Different scales | A feature with big numbers outweighs features with small numbers | Standardise all features |
| Odd rows | The national total row, and 3 Dublin areas with people but 0 households, would give wrong or missing values | Remove them and record it in the log |

## 1.7 Initial Project Plan

| Phase | Main Activities | Target Week |
|---|---|---|
| 2. Data Understanding | Describe the two datasets, explore them with charts (e.g. density by urban/rural category) and check data quality (missing values, odd rows) | Week 9 |
| 3. Data Preparation | Join the files on GUID, remove the odd rows, build the 8 features, log-transform density, standardise, and fill in the preparation log | Weeks 9–10 |
| 4. Modelling | Choose k (elbow and silhouette), run K-means, and run one alternative to compare | Week 10 |
| 5. Evaluation | Check the 1.5 targets, describe each group, and compare with the CSO categories | Week 11  |
| 6. Deployment | Recommendations for planners, monitoring, limitations, and reflection | Week 11 |
| Paper and final checks | Write the 4-page paper and AI statement, Restart & Run All, and submit | Week 12 |

I am slightly behind the brief's milestones, so weeks 9 and 10 are the busiest. The risk responses from 1.6 happen in Phase 3 (odd rows, log-transform, standardise, dropping % car) and 4.1 (choosing k). The 1.5 targets are checked in Phase 5.

CRISP-DM is iterative, so this plan may change, for example going back to Data Preparation if the groups don't make sense. Any changes will be recorded with the reason.

### Why K-means
K-means fits this project because it is simple and fast on 18,919 rows, and it doesn't need labels, which I don't have. Each group's centre is just an average, so it is easy to describe to planners. The downside is that I have to choose k myself, which is why I'll use the elbow method and silhouette score.

### Baseline and Alternative
- **Baseline:** the CSO's 6 urban/rural categories. I will check if my groups add anything new beyond these (set up in 1.5).
- **Alternative:** hierarchical clustering on a sample of small areas, to see if it finds similar groups to K-means. Details are in 4.3.

# Phase 2: Data Understanding

## 2.1 Initial Data Collection Report

### Sources
- The SAPS file containing 18,919 small areas in ireland was collected by the CSO. It is an official census dataset collected by the CSO and published by the CSO as open data. The licence is CC by 4.0 and was originally accessed on 3rd Nov 2026

- The boundaries file was published by Tailte Eireann and was also collected by Tailte Eireann. It's license falls under the CC by 4.0. The dataset was originally accessed on 3rd Nov 2026

- The glossary can be found on the same page as the SAPS file. It provides the required column information to understand the data more. The license is CC by 4.0. THe dataset was originally accessed on 3rd Nov 2026

### How the data was collected
The Irish cencus is a count of every person and household in the state officially ran by the CSO. All the data was collected on census night (3rd April, 2022). The data is only a snapshot of the one night in 2022 so information like vacancy is a point in time measurement. The data was then summarized for each small area which is what became of the SAPS 2022 data set we are working with.

The boundaries file was created by Tailte Eireann with the assistance of the CSO. This makes the census area have a matching shape on the map and therefore a size.

### Scope
The dataset we are working with covers the republic of ireland and does not include northern ireland. I am working with 18,919 small areas that has roughly 80-120 homes each. The dataset is a snapshot from April 3rd 2022. There are roughly 15 census themes but I am going to be primarily working with Housing and commuting (themes 6 and 11). I will also be using land area from the Tailte Eireann boundaries file.

### Files acquired
| File | Publisher | Format and size | Rows x columns | What it contains |
|---|---|---|---|---|
| SAPS_2022_Small_Area_UR_171024.csv| CSO | CSV |18,920 × 795 | Census counts per small area + urban/rural category |
| Small_Area_National_Statistical_Boundaries_2022_..._.csv | Tailte Éireann | CSV | 18,919 × 28 | Census counts per small area + urban/rural category |
| Glossary_Saps_2022_REVISED_21102024.xlsx | CSO | Excel | 790 × 4 | Meaning for the columns on the census SAPS file |

### How the files relate
The glossary provides column codes/headings for the CSO SAPS small area file. The census and the boundaries datasets join together on the small area ID (GUID = SA_GUID_2022). The census file has 1 extra row compared to the boundaries file. This row is for the national total which will be cut due it it's irrelevance.

### Limitations
The boundary dataset can contain things like fields, roads, rivers etc as it covereds a general small area. There is no data for individual households so density can't be measured with absolute accuracy. The data is a snapshot of a single point in time (april 3rd 2022) so density over time isn't meaasured.


In [8]:
# Load the three raw files (raw data is never edited)
census = pd.read_csv(SAPS_FILE, low_memory=False, dtype={"GEOGID": str, "GEOGDESC": str})
boundary = pd.read_csv(BOUNDARY_FILE, low_memory=False)
glossary = pd.read_excel(GLOSSARY_FILE)

print("Census   :", census.shape)
print("Boundary :", boundary.shape)
print("Glossary :", glossary.shape)

Census   : (18920, 795)
Boundary : (18919, 28)
Glossary : (790, 4)


## 2.2 Data Description Report



This section describes the size and structure of the data, what the columns mean and what values are expected

In [9]:
# Check how big both files are, what type each column is, and the first few rows
print("Census rows x columns  :", census.shape)
print("Boundary rows x columns:", boundary.shape)
print(census.dtypes.value_counts())
census[["GUID", "GEOGID", "GEOGDESC", "UR_Category_Desc", "T1_1AGETT", "T6_1_TH"]].head()

Census rows x columns  : (18920, 795)
Boundary rows x columns: (18919, 28)
int64      790
str          4
float64      1
Name: count, dtype: int64


,GUID,GEOGID,GEOGDESC,UR_Category_Desc,T1_1AGETT,T6_1_TH
0,4c07d11e-11d3-851d-e053-ca3ca8c0ca7f,017001001,017001001,4. Rural areas with high urban influence,419,137
1,4c07d11e-123a-851d-e053-ca3ca8c0ca7f,017002001,017002001,4. Rural areas with high urban influence,345,118
2,4c07d11e-14b1-851d-e053-ca3ca8c0ca7f,017002002,017002002,3. Independent urban towns,338,119
3,bf640964-28f3-4ccf-a610-04685d80ea2e,017002003/01,017002003/01,4. Rural areas with high urban influence,439,158
4,4c07d11d-f709-851d-e053-ca3ca8c0ca7f,017003001,017003001,5. Rural areas with moderate urban influence,255,91


Each row in both files is one small area. The two files link up using the small area ID. It's called GUID in the census file and SA_GUID_2022 in the boundary file.

GEOGID can't be used to link them. Some small areas were joined together, so their code looks different, like 017002003/01. The boundary file doesn't have a GEOGDESC column at all.

Most columns (790) are whole numbers. These are counts, like how many people live in the area or how many homes are flats. They aren't codes. 4 columns are text, such as the ID and the area name.

One column (UR_Category) has decimals (float64). That's because one row has a blank in it. This is the national total row, which I'll remove later.

In [10]:
# Check each ID shows up once, and count the areas in each CSO group
print("Census GUID unique?  ", census["GUID"].is_unique)
print("Boundary GUID unique?", boundary["SA_GUID_2022"].is_unique)
print(census["UR_Category_Desc"].value_counts(dropna=False))

Census GUID unique?   True
Boundary GUID unique? True
UR_Category_Desc
1. Cities                                       6250
3. Independent urban towns                      3325
4. Rural areas with high urban influence        2401
5. Rural areas with moderate urban influence    2362
6. Highly rural/remote areas                    2357
2. Satellite urban towns                        2224
NaN                                                1
Name: count, dtype: int64


Both ID columns are unique. Each small area shows up only once in each file, so every row will match exactly one row in the other file.

The output also shows how many small areas are in each CSO urban/rural group. Cities is the biggest (6,250). Satellite urban towns is the smallest (2,224). One row is blank (NaN). That's the national total row again, and I'll remove it in Phase 3.

I use this CSO group in two different ways:
- To compare my groups against (1.5).
- To tell if a group is close to towns (1.4).

In [11]:
# List the boundary columns and show land area in km² (it starts in square metres)
print(boundary.columns.tolist())
(boundary["Shape__Area"] / 1_000_000).describe().round(3)

['OBJECTID', 'SA_GUID_2016', 'SA_GUID_2022', 'SA_PUB2011', 'SA_PUB2016', 'SA_PUB2022', 'SA_GEOGID_2022', 'SA_CHANGE_CODE', 'SA_URBAN_AREA_FLAG', 'SA_URBAN_AREA_NAME', 'SA_NUTS1', 'SA_NUTS1_NAME', 'SA_NUTS2', 'SA_NUTS2_NAME', 'SA_NUTS3', 'SA_NUTS3_NAME', 'ED_GUID', 'ED_OFFICIAL', 'ED_ENGLISH', 'ED_GAEILGE', 'ED_ID_STR', 'ED_PART_COUNT', 'COUNTY_CODE', 'COUNTY_ENGLISH', 'COUNTY_GAEILGE', 'CSO_LEA', 'Shape__Area', 'Shape__Length']


count    18919.000
mean         3.714
std          7.412
min          0.002
25%          0.049
50%          0.132
75%          5.661
max        163.253
Name: Shape__Area, dtype: float64

This file has the land area, county and region names. These will help when I group areas by county. The land area is in square metres, so I changed it to km².

The sizes vary a lot. The smallest area is 0.002 km² and the biggest is 163 km². That's because small areas are based on the number of homes, not on size. So city areas are tiny and rural ones are big.

The average (3.7 km²) is much bigger than the middle value (0.13 km²). Most areas are small, but a few huge rural ones pull the average up. Homes per km² uses this land area, so it will have the same problem. That's why I plan to use the log transform from 1.6, which shrinks the very big values.

In [15]:
# Make a data dictionary: the meaning of the 17 census columns I need
glossary.columns = ["Theme", "Table", "Column", "Description"]
source_cols = ["T1_1AGETT", "T6_1_TH", "T6_1_TP", "T6_1_FA_H",
               "T6_2_TH", "T6_2_11_15H", "T6_2_16LH",
               "T6_3_TH", "T6_3_RPLH", "T6_3_RLAH", "T6_3_RVCHBH",
               "T6_8_T", "T6_8_OVD", "T6_8_UHH",
               "T11_1_TW", "T11_1_BUW", "T11_1_TDLW"]        # the 17 raw census columns my features use
data_dictionary = glossary[glossary["Column"].isin(source_cols)][["Column", "Description", "Table"]]
data_dictionary

,Column,Description,Table
104,T1_1AGETT,Total,Table 1: Population aged 0 - 19 by sex and yea...
349,T6_1_FA_H,Flat/Apartment (No. of households),Table 1: Private households by type of accommo...
352,T6_1_TH,Total (No. of households),Table 1: Private households by type of accommo...
357,T6_1_TP,Total (No. of persons),Table 1: Private households by type of accommo...
366,T6_2_11_15H,2011 - 2015 (No. of households),Table 2: Permanent private households by year ...
367,T6_2_16LH,2016 or later (No. of households),Table 2: Permanent private households by year ...
369,T6_2_TH,Total (No. of households),Table 2: Permanent private households by year ...
384,T6_3_RPLH,Rented from private landlord (No. of households),Table 3: Permanent private households by type ...
385,T6_3_RLAH,Rented from Local Authority (No. of households),Table 3: Permanent private households by type ...
386,T6_3_RVCHBH,Rented from voluntary/co-operative housing bod...,Table 3: Permanent private households by type ...


This table is my data dictionary. It lists the 17 census columns I need to build my 8 features. The meanings come from the CSO glossary.

The tables don't all count the same homes. Table 1 counts all households. Tables 2 and 3 only count permanent ones. You can see the gap in the totals in the next output (1,841,152 vs 1,836,728).

In [16]:
# Show the smallest, middle and biggest value of each column to check they make sense
census[source_cols].describe().T[["min", "50%", "max"]]

,min,50%,max
T1_1AGETT,37.0,259.0,5149139.0
T6_1_TH,0.0,94.0,1841152.0
T6_1_TP,0.0,256.0,5046681.0
T6_1_FA_H,0.0,0.0,239299.0
T6_2_TH,0.0,94.0,1836728.0
T6_2_11_15H,0.0,1.0,49160.0
T6_2_16LH,0.0,1.0,93128.0
T6_3_TH,0.0,94.0,1836728.0
T6_3_RPLH,0.0,10.0,330632.0
T6_3_RLAH,0.0,2.0,153192.0


None of the numbers are below 0. The middle values look normal for one small area. For example, the middle small area has 259 people and 94 households.

The biggest values can't be right for one small area. The biggest population is 5,149,139, which is the whole country. That's the national total row again, so it's one more reason to remove it.

### Inputs and target
The inputs are the 8 features from 1.4:
- Homes per km² (housing density)
- % Flats/Apartments
- % built since 2011
- Vacancy rate
- Holiday home rate
- People per household
- % renting
- % public transport to work

There's no target. Clustering just groups similar areas together, so there's no "right answer" column to learn from. The CSO urban/rural group isn't an input. I'll only use it after clustering, to compare with my groups (5.3).

## 2.3 Data Exploration Report

## 2.4 Data Quality Report

# Phase 3: Data Preparation

## 3.1 Selection

## 3.2 Cleaning

## 3.3 Integration (Join on GUID)

## 3.4 Feature Construction

## 3.5 Transformation and Scaling

## 3.6 Embedded Preparation Log

## 3.7 Validation (Before/After)

# Phase 4: Modelling

## 4.1 Choosing k (Elbow and Silhouette)

## 4.2 K-Means Model

## 4.3 Alternative for Comparison

# Phase 5: Evaluation

## 5.1 Technical Results

## 5.2 Cluster Profiles

## 5.3 Comparison with CSO Urban/Rural Categories

## 5.4 Comparison with Success Criteria

## 5.5 Decision: Proceed, Revise or Reject

# Phase 6: Deployment and Recommendations

## 6.1 Intended Use and Users

## 6.2 Monitoring and Governance

## 6.3 Future Work

# Personal Reflection

# Responsible AI Collaboration Statement (300–500 words)